## Extract

In [1]:
import pandas as pd
from pathlib import Path

df_vegetation = pd.read_pickle("data/raw/pickle/df_vegetation_2016_2026.pkl")

# Optional: standardize column names
df_vegetation.columns = df_vegetation.columns.str.lower()

# View data
print(df_vegetation.head())
print(df_vegetation.shape)

   district_id district_name  state_id      state_name boundary_version  \
0        17892      Azamgarh     70081   Uttar Pradesh        GAUL_2015   
1        17868    Dharmapuri      1508      Tamil Nadu        GAUL_2015   
2        17756    West Nimar     70079  Madhya Pradesh        GAUL_2015   
3        17589        Nagaon      1487           Assam        GAUL_2015   
4        17565    Srikakulam      1485  Andhra Pradesh        GAUL_2015   

         date  period_end  year  month     source_dataset  ...  \
0  2024-11-16  2024-12-02  2024     11  MODIS/061/MOD13Q1  ...   
1  2025-06-26  2025-07-12  2025      6  MODIS/061/MOD13Q1  ...   
2  2025-12-03  2025-12-19  2025     12  MODIS/061/MOD13Q1  ...   
3  2024-11-16  2024-12-02  2024     11  MODIS/061/MOD13Q1  ...   
4  2026-02-02  2026-02-18  2026      2  MODIS/061/MOD13Q1  ...   

   valid_crop_area_km2  valid_crop_fraction  crop_mask_year  \
0            20.252766             0.005278          2023.0   
1           923.512498    

## Transform

In [2]:
df_vegetation["state_name"]= df_vegetation["state_name"].str.lower().str.replace(" ", "_")
df_vegetation["district_name"]= df_vegetation["district_name"].str.lower().str.replace(" ", "_")
df_vegetation

,district_id,district_name,state_id,state_name,boundary_version,date,period_end,year,month,source_dataset,...,valid_crop_area_km2,valid_crop_fraction,crop_mask_year,quality_status,previous_composite_date,composite_gap_days,ndvi_lag_1period,evi_lag_1period,ndvi_change,ndvi_slope_per_day
0,17892,azamgarh,70081,uttar_pradesh,GAUL_2015,2024-11-16,2024-12-02,2024,11,MODIS/061/MOD13Q1,...,20.252766,0.005278,2023.0,low_valid_crop_fraction,2024-10-31,16.0,NaN,NaN,NaN,NaN
1,17868,dharmapuri,1508,tamil_nadu,GAUL_2015,2025-06-26,2025-07-12,2025,6,MODIS/061/MOD13Q1,...,923.512498,0.137893,2024.0,low_valid_crop_fraction,2025-06-10,16.0,NaN,NaN,NaN,NaN
2,17756,west_nimar,70079,madhya_pradesh,GAUL_2015,2025-12-03,2025-12-19,2025,12,MODIS/061/MOD13Q1,...,5510.744283,0.722399,2024.0,valid,2025-11-17,16.0,NaN,NaN,NaN,NaN
3,17589,nagaon,1487,assam,GAUL_2015,2024-11-16,2024-12-02,2024,11,MODIS/061/MOD13Q1,...,2402.394854,0.990117,2023.0,valid,2024-10-31,16.0,0.636606,0.378194,-0.075447,-0.004715
4,17565,srikakulam,1485,andhra_pradesh,GAUL_2015,2026-02-02,2026-02-18,2026,2,MODIS/061/MOD13Q1,...,1262.716656,0.272212,2024.0,valid,2026-01-17,16.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52711,17888,agra,70081,uttar_pradesh,GAUL_2015,2021-05-09,2021-05-25,2021,5,MODIS/061/MOD13Q1,...,3320.980760,0.999399,2020.0,valid,2021-04-23,16.0,0.225445,0.157105,0.068675,0.004292
52712,17888,agra,70081,uttar_pradesh,GAUL_2015,2023-08-13,2023-08-29,2023,8,MODIS/061/MOD13Q1,...,471.182284,0.141289,2022.0,low_valid_crop_fraction,2023-07-28,16.0,NaN,NaN,NaN,NaN
52713,17837,barmer,1506,rajasthan,GAUL_2015,2020-03-21,2020-04-06,2020,3,MODIS/061/MOD13Q1,...,2734.134189,0.997565,2019.0,valid,2020-03-05,16.0,0.230255,0.156444,-0.027548,-0.001722
52714,70276,sant_ravi_das_nagar,70081,uttar_pradesh,GAUL_2015,2021-03-06,2021-03-22,2021,3,MODIS/061/MOD13Q1,...,370.482730,0.382070,2020.0,valid,2021-02-18,16.0,0.621845,0.427445,-0.100976,-0.006311


In [9]:
# Step 1: Create the lookup mapping by grouping by district, year, and month
ndvi_lookup = (
    df_vegetation
    .groupby(["district_name", "year", "month"])["ndvi_mean"]
    .mean()
)

evi_lookup = (
    df_vegetation
    .groupby(["district_name", "year", "month"])["evi_mean"]
    .mean()
)

# Step 2: Initialize df_vegetation_filtered as a copy of df_vegetation
df_vegetation_filtered = df_vegetation.copy()

# Step 3: Map the aggregated NDVI values back to df_vegetation_filtered
df_vegetation_filtered["ndvi"] = pd.MultiIndex.from_frame(
    df_vegetation_filtered[["district_name", "year", "month"]]
).map(ndvi_lookup)

# Step 4: Map the aggregated EVI values back to df_vegetation_filtered
df_vegetation_filtered["evi"] = pd.MultiIndex.from_frame(
    df_vegetation_filtered[["district_name", "year", "month"]]
).map(evi_lookup)

df_vegetation_filtered

,district_id,district_name,state_id,state_name,boundary_version,date,period_end,year,month,source_dataset,...,crop_mask_year,quality_status,previous_composite_date,composite_gap_days,ndvi_lag_1period,evi_lag_1period,ndvi_change,ndvi_slope_per_day,ndvi,evi
0,17892,azamgarh,70081,uttar_pradesh,GAUL_2015,2024-11-16,2024-12-02,2024,11,MODIS/061/MOD13Q1,...,2023.0,low_valid_crop_fraction,2024-10-31,16.0,NaN,NaN,NaN,NaN,NaN,NaN
1,17868,dharmapuri,1508,tamil_nadu,GAUL_2015,2025-06-26,2025-07-12,2025,6,MODIS/061/MOD13Q1,...,2024.0,low_valid_crop_fraction,2025-06-10,16.0,NaN,NaN,NaN,NaN,NaN,NaN
2,17756,west_nimar,70079,madhya_pradesh,GAUL_2015,2025-12-03,2025-12-19,2025,12,MODIS/061/MOD13Q1,...,2024.0,valid,2025-11-17,16.0,NaN,NaN,NaN,NaN,0.500741,0.261949
3,17589,nagaon,1487,assam,GAUL_2015,2024-11-16,2024-12-02,2024,11,MODIS/061/MOD13Q1,...,2023.0,valid,2024-10-31,16.0,0.636606,0.378194,-0.075447,-0.004715,0.561159,0.320175
4,17565,srikakulam,1485,andhra_pradesh,GAUL_2015,2026-02-02,2026-02-18,2026,2,MODIS/061/MOD13Q1,...,2024.0,valid,2026-01-17,16.0,NaN,NaN,NaN,NaN,0.467432,0.277914
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52711,17888,agra,70081,uttar_pradesh,GAUL_2015,2021-05-09,2021-05-25,2021,5,MODIS/061/MOD13Q1,...,2020.0,valid,2021-04-23,16.0,0.225445,0.157105,0.068675,0.004292,0.305701,0.214211
52712,17888,agra,70081,uttar_pradesh,GAUL_2015,2023-08-13,2023-08-29,2023,8,MODIS/061/MOD13Q1,...,2022.0,low_valid_crop_fraction,2023-07-28,16.0,NaN,NaN,NaN,NaN,0.559330,0.410284
52713,17837,barmer,1506,rajasthan,GAUL_2015,2020-03-21,2020-04-06,2020,3,MODIS/061/MOD13Q1,...,2019.0,valid,2020-03-05,16.0,0.230255,0.156444,-0.027548,-0.001722,0.216482,0.149700
52714,70276,sant_ravi_das_nagar,70081,uttar_pradesh,GAUL_2015,2021-03-06,2021-03-22,2021,3,MODIS/061/MOD13Q1,...,2020.0,valid,2021-02-18,16.0,0.621845,0.427445,-0.100976,-0.006311,0.425397,0.298425


In [5]:
df_vegetation_filtered["ndvi"].isnull().sum()  # Check for any null values in the new 'ndvi' column

np.int64(46648)

In [16]:
df_vegetation_filtered["ndvi"].max()

0.8659786369357881

In [17]:
df_vegetation_filtered["ndvi"].min()

0.1223937385301611

In [8]:
df_vegetation_filtered["evi_mean"].isnull().sum()

np.int64(19568)

In [14]:
df_vegetation_filtered["evi_mean"].max()

0.7707881273866216

In [15]:
# Calculate Max - Min
df_vegetation_filtered["evi_mean"].min()

0.0758168634456552

In [10]:
df_vegetation_filtered=df_vegetation_filtered[df_vegetation_filtered['district_name'].notnull()]
df_vegetation_filtered=df_vegetation_filtered[df_vegetation_filtered['state_name'].notnull()]
df_vegetation_filtered=df_vegetation_filtered[df_vegetation_filtered['date'].notnull()]
df_vegetation_filtered=df_vegetation_filtered[df_vegetation_filtered['ndvi'].notnull()]
df_vegetation_filtered=df_vegetation_filtered[df_vegetation_filtered['evi'].notnull()]
df_vegetation_filtered

,district_id,district_name,state_id,state_name,boundary_version,date,period_end,year,month,source_dataset,...,crop_mask_year,quality_status,previous_composite_date,composite_gap_days,ndvi_lag_1period,evi_lag_1period,ndvi_change,ndvi_slope_per_day,ndvi,evi
2,17756,west_nimar,70079,madhya_pradesh,GAUL_2015,2025-12-03,2025-12-19,2025,12,MODIS/061/MOD13Q1,...,2024.0,valid,2025-11-17,16.0,NaN,NaN,NaN,NaN,0.500741,0.261949
3,17589,nagaon,1487,assam,GAUL_2015,2024-11-16,2024-12-02,2024,11,MODIS/061/MOD13Q1,...,2023.0,valid,2024-10-31,16.0,0.636606,0.378194,-0.075447,-0.004715,0.561159,0.320175
4,17565,srikakulam,1485,andhra_pradesh,GAUL_2015,2026-02-02,2026-02-18,2026,2,MODIS/061/MOD13Q1,...,2024.0,valid,2026-01-17,16.0,NaN,NaN,NaN,NaN,0.467432,0.277914
5,17589,nagaon,1487,assam,GAUL_2015,2024-10-15,2024-10-31,2024,10,MODIS/061/MOD13Q1,...,2023.0,low_valid_crop_fraction,2024-09-29,16.0,NaN,NaN,NaN,NaN,0.636606,0.378194
8,17856,sawai_madhopur,1506,rajasthan,GAUL_2015,2024-02-02,2024-02-18,2024,2,MODIS/061/MOD13Q1,...,2023.0,valid,2024-01-17,16.0,NaN,NaN,NaN,NaN,0.579402,0.381551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52711,17888,agra,70081,uttar_pradesh,GAUL_2015,2021-05-09,2021-05-25,2021,5,MODIS/061/MOD13Q1,...,2020.0,valid,2021-04-23,16.0,0.225445,0.157105,0.068675,0.004292,0.305701,0.214211
52712,17888,agra,70081,uttar_pradesh,GAUL_2015,2023-08-13,2023-08-29,2023,8,MODIS/061/MOD13Q1,...,2022.0,low_valid_crop_fraction,2023-07-28,16.0,NaN,NaN,NaN,NaN,0.559330,0.410284
52713,17837,barmer,1506,rajasthan,GAUL_2015,2020-03-21,2020-04-06,2020,3,MODIS/061/MOD13Q1,...,2019.0,valid,2020-03-05,16.0,0.230255,0.156444,-0.027548,-0.001722,0.216482,0.149700
52714,70276,sant_ravi_das_nagar,70081,uttar_pradesh,GAUL_2015,2021-03-06,2021-03-22,2021,3,MODIS/061/MOD13Q1,...,2020.0,valid,2021-02-18,16.0,0.621845,0.427445,-0.100976,-0.006311,0.425397,0.298425


## Load

In [ ]:
df_vegetation_filtered.to_pickle(r"data\raw\processed_ram\fact_vegetation.pkl")